# 1. Data Pipeline for AI 

## 1.1 Load Performance Metrics Dataset 

In [1]:
import pandas as pd

metrics_path = r"A:\pc\Desktop\Code\data science\sadop\Data\slow_query_metrics.csv"
df = pd.read_csv(metrics_path)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (19975, 14)


,query,query_time,rows_returned,has_sum,has_group_by,has_where,tables_count,query_length,cpu_usage,estimated_rows,uses_index,full_table_scan,uses_filesort,uses_temp_table
0,SELECT *\nFROM user\nWHERE user_id IN (\n S...,0.850983,16520,0,0,1,1,189,25.0,54923,1,1,0,0
1,"SELECT u.user_id,\n (\n SELECT S...",0.745494,20000,1,0,1,2,217,0.0,20207,1,0,0,0
2,SELECT *\nFROM user u\nWHERE EXISTS (\n SEL...,0.574111,16121,0,0,1,2,184,0.0,54923,1,1,0,0
3,"SELECT u.user_id, t.transaction_date, t.amount...",1.684971,250000,0,0,0,3,185,0.0,34725,1,0,1,1
4,"SELECT u.user_id,\n (\n SELECT S...",0.599545,20000,1,0,1,2,217,0.0,20207,1,0,0,0


## 1.2 Inspect Feature Distributions

In [2]:
df.describe()

,query_time,rows_returned,has_sum,has_group_by,has_where,tables_count,query_length,cpu_usage,estimated_rows,uses_index,full_table_scan,uses_filesort,uses_temp_table
count,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000
mean,0.878726,42029.306783,0.250513,0.371464,0.561752,2.247209,159.240551,8.006894,132055.045457,0.878098,0.560901,0.127559,0.440801
std,1.517166,75354.039549,0.433319,0.483208,0.496184,0.896666,53.536735,16.997823,118991.382014,0.327181,0.496290,0.333607,0.496496
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,47.000000,0.000000,9.000000,0.000000,0.000000,0.000000,0.000000
25%,0.086555,2669.000000,0.000000,0.000000,0.000000,2.000000,151.000000,0.000000,34724.000000,1.000000,0.000000,0.000000,0.000000
50%,0.495865,16520.000000,0.000000,0.000000,1.000000,2.000000,183.000000,0.000000,54915.000000,1.000000,1.000000,0.000000,0.000000
75%,0.719669,20000.000000,1.000000,1.000000,1.000000,3.000000,198.000000,12.500000,249887.000000,1.000000,1.000000,0.000000,1.000000
max,9.993443,250000.000000,1.000000,1.000000,1.000000,4.000000,227.000000,100.000000,295636.000000,1.000000,1.000000,1.000000,1.000000


## 1.3 Define ML Features

In [5]:
FEATURE_COLUMNS = [
   
    "query_time",
    "rows_returned",
    "has_sum",
    "has_group_by",
    "has_where",
    "tables_count",
    "query_length",
    "cpu_usage",  
    "estimated_rows",
    "uses_index",
    "full_table_scan",
    "uses_filesort",
    "uses_temp_table",
]

X = df[FEATURE_COLUMNS]

# Fill NaN in estimated_rows (EXPLAIN couldn't compute it) with 0
X = X.fillna({"estimated_rows": 0})

print("Feature matrix shape:", X.shape)
X.describe()


Feature matrix shape: (19975, 13)


,query_time,rows_returned,has_sum,has_group_by,has_where,tables_count,query_length,cpu_usage,estimated_rows,uses_index,full_table_scan,uses_filesort,uses_temp_table
count,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000
mean,0.878726,42029.306783,0.250513,0.371464,0.561752,2.247209,159.240551,8.006894,132055.045457,0.878098,0.560901,0.127559,0.440801
std,1.517166,75354.039549,0.433319,0.483208,0.496184,0.896666,53.536735,16.997823,118991.382014,0.327181,0.496290,0.333607,0.496496
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,47.000000,0.000000,9.000000,0.000000,0.000000,0.000000,0.000000
25%,0.086555,2669.000000,0.000000,0.000000,0.000000,2.000000,151.000000,0.000000,34724.000000,1.000000,0.000000,0.000000,0.000000
50%,0.495865,16520.000000,0.000000,0.000000,1.000000,2.000000,183.000000,0.000000,54915.000000,1.000000,1.000000,0.000000,0.000000
75%,0.719669,20000.000000,1.000000,1.000000,1.000000,3.000000,198.000000,12.500000,249887.000000,1.000000,1.000000,0.000000,1.000000
max,9.993443,250000.000000,1.000000,1.000000,1.000000,4.000000,227.000000,100.000000,295636.000000,1.000000,1.000000,1.000000,1.000000


## 1.4 Create Performance Labels 

In [10]:
SLOW_QUERY_THRESHOLD = 0.6
df["is_slow"] = (df["query_time"] >= SLOW_QUERY_THRESHOLD).astype(int)
y = df["is_slow"]
df[["query_time", "is_slow"]].head()

,query_time,is_slow
0,0.850983,1
1,0.745494,1
2,0.574111,0
3,1.684971,1
4,0.599545,0


## 1.5 Check Class Balance

In [12]:
print("Class distribution:")
print(df["is_slow"].value_counts())
print()
print("Class ratio (normalized):")
print(df["is_slow"].value_counts(normalize=True).round(3))


Class distribution:
is_slow
0    11200
1     8775
Name: count, dtype: int64

Class ratio (normalized):
is_slow
0    0.561
1    0.439
Name: proportion, dtype: float64


## 1.6 Save Feature Dataset 

In [56]:
feature_dataset_path = r"C:\Users\pc\data science\SADOP\data\ml_features.csv"
df.to_csv(feature_dataset_path, index=False)

print(f"ML-ready dataset saved to {feature_dataset_path}")

ML-ready dataset saved to C:\Users\pc\data science\SADOP\data\ml_features.csv


In [13]:
feature_dataset_path = r"A:\pc\Desktop\Code\data science\sadop\Data\ml_features.csv"
df.to_csv(feature_dataset_path, index=False)
print(f"✅ ML-ready dataset saved → {feature_dataset_path}")
print(f"   Shape  : {df.shape}")
print(f"   Slow   : {df['is_slow'].sum():,}  ({df['is_slow'].mean()*100:.1f} %)")
print(f"   Not slow: {(df['is_slow']==0).sum():,}  ({(df['is_slow']==0).mean()*100:.1f} %)")


✅ ML-ready dataset saved → A:\pc\Desktop\Code\data science\sadop\Data\ml_features.csv
   Shape  : (19975, 15)
   Slow   : 8,775  (43.9 %)
   Not slow: 11,200  (56.1 %)


# 1.7 Full Stats Overview

In [14]:
df[FEATURE_COLUMNS + ["is_slow"]].describe()

,query_time,rows_returned,has_sum,has_group_by,has_where,tables_count,query_length,cpu_usage,estimated_rows,uses_index,full_table_scan,uses_filesort,uses_temp_table,is_slow
count,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000,19975.000000
mean,0.878726,42029.306783,0.250513,0.371464,0.561752,2.247209,159.240551,8.006894,132055.045457,0.878098,0.560901,0.127559,0.440801,0.439299
std,1.517166,75354.039549,0.433319,0.483208,0.496184,0.896666,53.536735,16.997823,118991.382014,0.327181,0.496290,0.333607,0.496496,0.496314
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,47.000000,0.000000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.086555,2669.000000,0.000000,0.000000,0.000000,2.000000,151.000000,0.000000,34724.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,0.495865,16520.000000,0.000000,0.000000,1.000000,2.000000,183.000000,0.000000,54915.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,0.719669,20000.000000,1.000000,1.000000,1.000000,3.000000,198.000000,12.500000,249887.000000,1.000000,1.000000,0.000000,1.000000,1.000000
max,9.993443,250000.000000,1.000000,1.000000,1.000000,4.000000,227.000000,100.000000,295636.000000,1.000000,1.000000,1.000000,1.000000,1.000000
